# Regularization — Linear vs Ridge vs Lasso vs Elastic Net

Tunes `alpha` (and `l1_ratio` for Elastic Net) by 5-fold CV on the **training** set, and inspects coefficient shrinkage. Shows when regularization helps vs hurts.

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from src.preprocessing import prepare_data
from src.models import make_model, display_name
from src.tuning import regularization_search, best_cv_rmse, REG_GRIDS
from src.evaluation import evaluate
from src import config

In [ ]:
data = prepare_data()
print('features:', len(data['feature_names']))

### Unregularized reference

In [ ]:
lin = make_model('linear_regression', data['target_transform'])
res, _, _ = evaluate(lin, data, 'Linear Regression', 'linear_regression')
print(f"Linear  CV RMSE Rs{res['cv']['RMSE_mean']:,.0f}  val RMSE Rs{res['val']['RMSE']:,.0f}")

### Tuned Ridge / Lasso / Elastic Net

In [ ]:
rows=[]
for key in ('ridge_regression','lasso_regression','elastic_net'):
    gs = regularization_search(key, data)
    best = {k.split('__')[-1]:v for k,v in gs.best_params_.items()}
    est = gs.best_estimator_.regressor_.named_steps['model']
    coef = np.ravel(getattr(est,'coef_',[]))
    near0 = int((np.abs(coef) < 1e-6).sum())
    rows.append({'model':display_name(key), **best, 'cv_rmse':round(best_cv_rmse(gs)),
                 'coef_total':len(coef), 'coef_near_zero':near0})
    print(f"{display_name(key):16s} best {best}  CV RMSE Rs{best_cv_rmse(gs):,.0f}  zeros {near0}/{len(coef)}")
reg = pd.DataFrame(rows); reg.to_csv(config.REPORTS/'regularization_summary.csv', index=False)
print('\nsaved reports/regularization_summary.csv'); reg

**Interpretation:** Ridge/Lasso constrain coefficients to curb variance. Lasso drives weak coefficients to exactly zero (feature selection); if many go to zero with little CV-RMSE loss, the model was over-parameterised. If regularization raises CV RMSE, the unpenalised fit was not overfitting.